In [2]:
import gymnasium as gym

In [3]:
e=gym.make('FrozenLake-v1')

In [4]:
e.observation_space

Discrete(16)

In [5]:
e.action_space

Discrete(4)

In [7]:
e.reset()

(0, {'prob': 1})

In [2]:
import wandb

In [5]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")

In [6]:
wandb.login(key=secret_value_0)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bhalaniakshat (bhalaniakshat-dwarkadas-j-sanghvi-college-of-engineering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from dataclasses import dataclass
from typing import List, Tuple

In [59]:
CONFIG = {
    "env_id": "FrozenLake-v1",
    "hidden_size": 64,
    "batch_size": 100,
    "percentile": 70,
    "lr": 0.01,
    "gamma": 0.99,
    "solve_bound": 0.8,
    "is_slippery": False}

In [43]:
class One(gym.ObservationWrapper):
    def __init__(self,env):
        super().__init__(env)
        self.observation_space=gym.spaces.Box(0.0,1.0,(env.observation_space.n,),dtype=np.float32)
    def observation(self,obs):
        res=np.copy(self.observation_space.low)
        res[obs]=1.0
        return res

In [11]:
@dataclass
class EpisodeStep:
    observation: np.ndarray
    action: int

In [12]:
@dataclass
class Episode:
    reward: float
    steps: List[EpisodeStep]

In [13]:
class Net(nn.Module):
    def __init__(self, obs_size: int, hidden_size: int, n_actions: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_actions)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [47]:
@torch.inference_mode()
def iterate_batches(env, net, device):
    batch = []
    episode_reward = 0.0
    episode_steps = []
    
    # Use 'env' (passed argument), not 'e'
    obs, _ = env.reset()
    
    while True:
        # 1. Convert observation to tensor
        obs_v = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        
        # 2. Get action probabilities from the network
        logits = net(obs_v)
        probs = torch.softmax(logits, dim=1)
        
        # 3. Sample an action
        m = Categorical(probs)
        action = m.sample().item()
        
        # 4. Step the environment
        next_obs, reward, terminated, truncated, _ = env.step(action)
        
        # 5. Accumulate rewards and steps
        episode_reward += reward
        episode_steps.append(EpisodeStep(observation=obs, action=action))
        
        # 6. Handle end of episode
        if terminated or truncated:
            # Save the completed episode
            batch.append(Episode(reward=episode_reward, steps=episode_steps))
            
            # Reset trackers for next episode
            episode_reward = 0.0
            episode_steps = []
            obs, _ = env.reset()
            
            # 7. If we have a full batch, yield it
            if len(batch) == CONFIG["batch_size"]:
                yield batch
                batch = []
        else:
            # Continue to next step in current episode
            obs = next_obs

In [63]:
def filter_batch(batch, percentile):
    disc_rewards = list(map(lambda s: s.reward * (CONFIG['gamma'] ** len(s.steps)), batch))
    reward_bound = np.percentile(disc_rewards, percentile)

    train_obs = []
    train_act = []
    elite_batch = []
    for example, discounted_reward in zip(batch, disc_rewards):
        if discounted_reward > reward_bound:
            train_obs.extend(map(lambda step: step.observation, example.steps))
            train_act.extend(map(lambda step: step.action, example.steps))
            elite_batch.append(example)

    return elite_batch, train_obs, train_act, reward_bound

In [64]:
run = wandb.init(project="frozenlake-cem", config=CONFIG)
device = "cuda" if torch.cuda.is_available() else "cpu"
    # Create Env
env = One(gym.make(CONFIG["env_id"]))
obs_size = env.observation_space.shape[0]
n_actions = env.action_space.n
net = Net(obs_size, CONFIG["hidden_size"], n_actions).to(device)
objective = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=CONFIG["lr"])

In [67]:
full_batch = [] 
device = "cuda" if torch.cuda.is_available() else "cpu"

# Note: Ensure iterate_batches matches the definition above
for iter_no, batch in enumerate(iterate_batches(env, net, device)):
    reward_mean = np.mean([s.reward for s in batch])
    
    # Filter the batch (combining current with previous elite episodes)
    # Ensure CONFIG["percentile"] is defined (e.g., 70)
    elite_batch, obs, acts, reward_bound = filter_batch(full_batch + batch, CONFIG["percentile"])
    
    if not elite_batch:
        if iter_no % 10 == 0:
            print(f"Iter {iter_no}: No successful episodes yet. Mean Reward: {reward_mean:.2f}")
        continue

    # Prepare training tensors
    obs_v = torch.as_tensor(np.array(obs), dtype=torch.float32, device=device)
    acts_v = torch.as_tensor(acts, dtype=torch.long, device=device)

    # Keep only the most recent successful episodes in the buffer
    full_batch = elite_batch[-500:]

    optimizer.zero_grad()
    action_scores_v = net(obs_v)
    loss_v = objective(action_scores_v, acts_v)
    loss_v.backward()
    optimizer.step()

    print(f"{iter_no}: loss={loss_v.item():.3f}, rew_mean={reward_mean:.2f}, bound={reward_bound:.2f}, elite_len={len(full_batch)}")
    
    wandb.log({
        "loss": loss_v.item(),
        "reward_mean": reward_mean,
        "reward_bound": reward_bound,
        "elite_episodes": len(full_batch)
    })

    if reward_mean > CONFIG["solve_bound"]:
        print("Solved!")
        break

0: loss=0.850, rew_mean=0.08, bound=0.00, elite_len=8
1: loss=0.819, rew_mean=0.09, bound=0.00, elite_len=17
2: loss=0.818, rew_mean=0.09, bound=0.00, elite_len=26
3: loss=0.811, rew_mean=0.11, bound=0.00, elite_len=37
4: loss=0.812, rew_mean=0.09, bound=0.74, elite_len=41
5: loss=0.794, rew_mean=0.09, bound=0.82, elite_len=41
6: loss=0.800, rew_mean=0.12, bound=0.83, elite_len=42
7: loss=0.769, rew_mean=0.08, bound=0.85, elite_len=41
8: loss=0.776, rew_mean=0.12, bound=0.87, elite_len=39
9: loss=0.771, rew_mean=0.09, bound=0.86, elite_len=41
10: loss=0.741, rew_mean=0.13, bound=0.88, elite_len=41
11: loss=0.732, rew_mean=0.12, bound=0.90, elite_len=31
12: loss=0.745, rew_mean=0.06, bound=0.00, elite_len=37
13: loss=0.687, rew_mean=0.15, bound=0.89, elite_len=41
14: loss=0.694, rew_mean=0.09, bound=0.90, elite_len=37
15: loss=0.698, rew_mean=0.14, bound=0.90, elite_len=39
16: loss=0.699, rew_mean=0.16, bound=0.91, elite_len=42
17: loss=0.725, rew_mean=0.11, bound=0.91, elite_len=28
18:

KeyboardInterrupt: 